In [1]:
from pathlib import Path
import urllib.request
import tarfile
import hashlib

data_dir = Path("/tmp/aclImdb")
archive = Path("/tmp/aclImdb_v1.tar.gz")
url = "https://huggingface.co/datasets/NolanChai/aclImdb_v1/resolve/main/aclImdb_v1.tar.gz"

if not archive.exists():
    partial = Path(str(archive) + ".part")
    urllib.request.urlretrieve(url, partial)
    partial.replace(archive)

checksum = hashlib.sha256(archive.read_bytes()).hexdigest()
assert checksum == "c40f74a18d3b61f90feba1e17730e0d38e8b97c05fde7008942e91923d1658fe"

if not data_dir.exists():
    with tarfile.open(archive, "r:gz") as tar:
        tar.extractall("/tmp", filter="data")

print("SHA256:", checksum)
print("Thư mục:", sorted(p.name for p in data_dir.iterdir()))

SHA256: c40f74a18d3b61f90feba1e17730e0d38e8b97c05fde7008942e91923d1658fe
Thư mục: ['README', 'imdb.vocab', 'imdbEr.txt', 'test', 'train']


In [2]:
import pandas as pd

rows = []

for split in ["train", "test"]:
    for category, label in [("neg", 0), ("pos", 1)]:
        for path in sorted((data_dir / split / category).glob("*.txt")):
            rows.append({
                "id": str(path.relative_to(data_dir)),
                "split": split,
                "text": path.read_text(encoding="utf-8"),
                "label": label
            })

df = pd.DataFrame(rows)

print("Số dòng, số cột:", df.shape)
display(df.head(10))
print("\nBài đánh giá đầu tiên:")
print(df.loc[0, "text"])

Số dòng, số cột: (50000, 4)


,id,split,text,label
0,train/neg/0_3.txt,train,Story of a man who has unnatural feelings for ...,0
1,train/neg/10000_4.txt,train,Airport '77 starts as a brand new luxury 747 p...,0
2,train/neg/10001_4.txt,train,This film lacked something I couldn't put my f...,0
3,train/neg/10002_1.txt,train,"Sorry everyone,,, I know this is supposed to b...",0
4,train/neg/10003_1.txt,train,When I was little my parents took me along to ...,0
5,train/neg/10004_3.txt,train,"""It appears that many critics find the idea of...",0
6,train/neg/10005_3.txt,train,The second attempt by a New York intellectual ...,0
7,train/neg/10006_4.txt,train,"I don't know who to blame, the timid writers o...",0
8,train/neg/10007_1.txt,train,This film is mediocre at best. Angie Harmon is...,0
9,train/neg/10008_2.txt,train,The film is bad. There is no other way to say ...,0



Bài đánh giá đầu tiên:
Story of a man who has unnatural feelings for a pig. Starts out with a opening scene that is a terrific example of absurd comedy. A formal orchestra audience is turned into an insane, violent mob by the crazy chantings of it's singers. Unfortunately it stays absurd the WHOLE time with no general narrative eventually making it just too off putting. Even those from the era should be turned off. The cryptic dialogue would make Shakespeare seem easy to a third grader. On a technical level it's better than you might think with some good cinematography by future great Vilmos Zsigmond. Future stars Sally Kirkland and Frederic Forrest can be seen briefly.


In [3]:
display(pd.crosstab(df["split"], df["label"]))

print("Giá trị thiếu:")
print(df.isna().sum())

print("Bài rỗng:", df["text"].str.strip().eq("").sum())
print("ID trùng:", df["id"].duplicated().sum())
print("Nội dung trùng:", df["text"].duplicated().sum())

label,0,1
split,,
test,12500,12500
train,12500,12500


Giá trị thiếu:
id       0
split    0
text     0
label    0
dtype: int64
Bài rỗng: 0
ID trùng: 0
Nội dung trùng: 418


In [4]:
print("Nội dung trùng trong từng tập:")
for split, part in df.groupby("split"):
    print(split, part["text"].duplicated().sum())

train_texts = set(df.loc[df["split"] == "train", "text"])
test_texts = set(df.loc[df["split"] == "test", "text"])

print("Nội dung xuất hiện ở cả train và test:",
      len(train_texts & test_texts))

duplicates = df[df["text"].duplicated(keep=False)]
print("Nội dung có nhiều nhãn:",
      (df.groupby("text")["label"].nunique() > 1).sum())

display(duplicates.sort_values("text").head(6))

print("Số từ mỗi bài:")
display(df["text"].str.split().str.len().describe())

Nội dung trùng trong từng tập:
test 199
train 96
Nội dung xuất hiện ở cả train và test: 123
Nội dung có nhiều nhãn: 0


,id,split,text,label
28814,test/neg/2183_2.txt,test,"""Go Fish"" garnered Rose Troche rightly or wron...",0
26084,test/neg/10977_2.txt,test,"""Go Fish"" garnered Rose Troche rightly or wron...",0
33153,test/neg/6089_3.txt,test,"""Three"" is a seriously dumb shipwreck movie. M...",0
30976,test/neg/4129_3.txt,test,"""Three"" is a seriously dumb shipwreck movie. M...",0
2552,train/neg/12298_4.txt,train,"""Witchery"" might just be the most incoherent a...",0
34165,test/neg/69_4.txt,test,"""Witchery"" might just be the most incoherent a...",0


Số từ mỗi bài:


count    50000.000000
mean       231.156940
std        171.343997
min          4.000000
25%        126.000000
50%        173.000000
75%        280.000000
max       2470.000000
Name: text, dtype: float64

In [6]:
import re
train = df[df["split"] == "train"]

patterns = {
    "Thẻ HTML": r"<[^>]+>",
    "Mã HTML": r"&(?:#\d+|#x[0-9a-fA-F]+|[a-zA-Z]+);",
    "Đường dẫn": r"https?://\S+|www\.\S+",
}

for name, pattern in patterns.items():
    mask = train["text"].str.contains(pattern, regex=True)
    print(f"\n{name}: {mask.sum()} bài")

    examples = train.loc[mask, "text"].head(2)
    for text in examples:
        match = re.search(pattern, text)
        start = max(0, match.start() - 60)
        end = min(len(text), match.end() + 60)
        print(repr(text[start:end]))


Thẻ HTML: 14667 bài
" survivor's as they await help with time fast running out...<br /><br />Also known under the slightly different tile Airport 1"
"alculated what he needed from the actors. I just don't know.<br /><br />But could it have been the screenplay? Just exactly wh"

Mã HTML: 7 bài
'If you are expecting to see a lot in the bath of Alt&#305;oklar, as it is promised by the tag line, you will be very d'
'b to really care.<br /><br />Eye Candy: Ashley Totin shows T&A; Evy Lutzky gets topless briefly; and Jennifer Hart shows he'

Đường dẫn: 95 bài
'e entertaining.<br /><br />- A review by Jamie Robert Ward (http://www.invocus.net)'
'ng out.<br /><br />** out of 4<br /><br />Roving Reviewer - www.geocities.com/paul_johnr'


In [7]:
import re

train = df[df["split"] == "train"].copy()

patterns = {
    "Thẻ HTML": r"<[^>]+>",
    "Mã HTML": r"&(?:#\d+|#x[0-9a-fA-F]+|[a-zA-Z]+);",
    "URL": r"https?://\S+|www\.\S+",
    "Email": r"\b[\w.+-]+@[\w.-]+\.[a-zA-Z]{2,}\b",
    "Khoảng trắng liên tiếp": r"\s{2,}",
    "Ký tự điều khiển": r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]",
    "Ký tự ngoài ASCII": r"[^\x00-\x7f]",
    "Số": r"\d",
    "Dấu câu lặp": r"[!?]{2,}",
    "Chữ viết hoa": r"\b[A-Z]{3,}\b",
    "Từ phủ định": r"\b(?:no|not|never|cannot)\b|\b\w+n['’]t\b",
}

stats = []

for name, pattern in patterns.items():
    mask = train["text"].str.contains(
        pattern, flags=re.IGNORECASE if name == "Từ phủ định" else 0,
        regex=True
    )
    stats.append({
        "Dấu hiệu": name,
        "Số bài": int(mask.sum()),
        "Tỷ lệ (%)": round(mask.mean() * 100, 2)
    })

display(pd.DataFrame(stats))

,Dấu hiệu,Số bài,Tỷ lệ (%)
0,Thẻ HTML,14667,58.67
1,Mã HTML,7,0.03
2,URL,95,0.38
3,Email,23,0.09
4,Khoảng trắng liên tiếp,216,0.86
5,Ký tự điều khiển,2,0.01
6,Ký tự ngoài ASCII,2347,9.39
7,Số,14164,56.66
8,Dấu câu lặp,2159,8.64
9,Chữ viết hoa,9445,37.78


In [8]:
for name, pattern in patterns.items():
    flags = re.IGNORECASE if name == "Từ phủ định" else 0
    mask = train["text"].str.contains(pattern, flags=flags, regex=True)

    print(f"\n{name}:")
    for text in train.loc[mask, "text"].head(2):
        match = re.search(pattern, text, flags=flags)
        start = max(0, match.start() - 70)
        end = min(len(text), match.end() + 70)
        print(repr(text[start:end]))

lengths = train["text"].str.split().str.len()

print("\n5 bài ngắn nhất:")
for idx in lengths.nsmallest(5).index:
    print(train.loc[idx, "id"], "|", lengths.loc[idx], "từ")
    print(train.loc[idx, "text"], "\n")

print("\n5 bài dài nhất:")
display(train.loc[lengths.nlargest(5).index, ["id", "label"]]
        .assign(word_count=lengths.nlargest(5)))


Thẻ HTML:
"nt for the survivor's as they await help with time fast running out...<br /><br />Also known under the slightly different tile Airport 1977 this s"
"r who miscalculated what he needed from the actors. I just don't know.<br /><br />But could it have been the screenplay? Just exactly who was the "

Mã HTML:
'If you are expecting to see a lot in the bath of Alt&#305;oklar, as it is promised by the tag line, you will be very disappointe'
'was to numb to really care.<br /><br />Eye Candy: Ashley Totin shows T&A; Evy Lutzky gets topless briefly; and Jennifer Hart shows her right ti'

URL:
'allow to be entertaining.<br /><br />- A review by Jamie Robert Ward (http://www.invocus.net)'
'rth checking out.<br /><br />** out of 4<br /><br />Roving Reviewer - www.geocities.com/paul_johnr'

Email:
' type to spoil a movie for anyone, but I INVITE anyone to email me at foxbarking@yahoo.com to ask for my opinion on this movie before they waste a dime on it. I'
"I agree with most if not all 

,id,label,word_count
14454,train/pos/1175_9.txt,1,2470
12549,train/pos/10044_9.txt,1,1839
18758,train/pos/4383_9.txt,1,1830
18417,train/pos/4076_10.txt,1,1723
16846,train/pos/2662_10.txt,1,1601


In [9]:
from collections import Counter
import unicodedata

train = df[df["split"] == "train"]

suspect_chars = Counter(
    char
    for text in train["text"]
    for char in text
    if unicodedata.category(char) == "Cc"
    and char not in "\n\r\t"
)

for char, count in suspect_chars.most_common():
    print(f"\nKý tự: {repr(char)} | U+{ord(char):04X} | Số lần: {count}")

    examples = train.loc[
        train["text"].str.contains(char, regex=False), "text"
    ].head(2)

    for text in examples:
        position = text.index(char)
        print(repr(text[max(0, position - 60):position + 61]))


Ký tự: '\x96' | U+0096 | Số lần: 1368
'nd three Superman-type villains, it started off with a bang \x96 the best Clark Kent to Superman transformations and rescue '
"e most hopelessly laughable 'horror' films I have ever seen \x96 in fact I still haven't recovered from the experience of wa"

Ký tự: '\x85' | U+0085 | Số lần: 707
' fact, they all but wrote the words: "Previously on Superman\x85" in S2RD. The special effects weren\'t great in either Part '
'cism, I will dignify the film now by explaining why it blows\x85 <br /><br />1. In a well-executed musical, the songs should'

Ký tự: '\x97' | U+0097 | Số lần: 283
'ing that wakes you up from your sleep while watching Tashan \x97 the mega-disappointing, mind-numbing new film at the cinema'
'e elements, a band of hostile Kiowa, an unscrupulous trader \x97 played by Donald Pleasance, here giving one of his most rid'

Ký tự: '\x91' | U+0091 | Số lần: 71
"ood fellas at Webster's Dictionary define Logophobia as the \x91fear of words'. I ma

In [10]:
import html
import re
import unicodedata

char_map = {}

for code in range(0x80, 0xA0):
    try:
        char_map[chr(code)] = bytes([code]).decode("cp1252")
    except UnicodeDecodeError:
        pass

def clean_text(text):
    text = re.sub(r"<[^>]+>", " ", text)
    text = html.unescape(text)

    text = "".join(char_map.get(char, char) for char in text)
    text = unicodedata.normalize("NFC", text)

    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"\b[\w.+-]+@[\w.-]+\.[a-zA-Z]{2,}\b", " ", text)

    text = "".join(
        " " if unicodedata.category(char) == "Cc" else char
        for char in text
    )
    return re.sub(r"\s+", " ", text).strip()

df["clean_text"] = df["text"].apply(clean_text)

example_ids = train.loc[
    train["text"].str.contains(r"[\x85\x96\x8e]", regex=True)
].head(3).index

for idx in example_ids:
    print("\nFile:", df.loc[idx, "id"])
    print("Trước:", df.loc[idx, "text"])
    print("Sau:", df.loc[idx, "clean_text"])

print("\nBài rỗng sau xử lý:", df["clean_text"].eq("").sum())
print("Số bài thay đổi:", df["text"].ne(df["clean_text"]).sum())


File: train/neg/10055_3.txt
Trước: I grew up on the 'Superman II' theatrical version ("S2T") and as a kid, I loved it more than Part I since not only did it contain more Superman and three Superman-type villains, it started off with a bang  the best Clark Kent to Superman transformations and rescue scenes. Kids no longer had to impatiently wait for Superman to appear on screen, as in part I. Now as an adult, I can see how the mighty had fallen with S2T (See: my review.) I've always heard of the back-story on how they prematurely and unjustifiably fired the original's director, Richard Donner from part II. (It must have been a rarity back then to film two separate movies simultaneously, now it's common: 'Back to the Future' and 'Matrix' 2 & 3 for example.) Unfortunately, after finally seeing the Richard Donner Cut (or, "S2RD") I still can't fully recommend it. Gone, was the great Superman change scene, the entire Paris rescue, as was the wonderful recap of part I in S2T's opening. In 

In [11]:
checks = {
    "Bài rỗng": df["clean_text"].str.strip().eq(""),
    "Thẻ HTML": df["clean_text"].str.contains(r"<[^>]+>", regex=True),
    "URL": df["clean_text"].str.contains(r"https?://\S+|www\.\S+", regex=True),
    "Ký tự điều khiển": df["clean_text"].apply(
        lambda text: any(unicodedata.category(c) == "Cc" for c in text)
    ),
}

for name, mask in checks.items():
    print(f"{name}: {mask.sum()}")
    if mask.any():
        display(df.loc[mask, ["id", "clean_text"]].head(2))

print("\nNội dung trùng sau làm sạch:")
for split, part in df.groupby("split"):
    print(split, part["clean_text"].duplicated().sum())

train_texts = set(df.loc[df["split"] == "train", "clean_text"])
test_texts = set(df.loc[df["split"] == "test", "clean_text"])

print("Nội dung chung train/test:", len(train_texts & test_texts))
print(
    "Nội dung có nhiều nhãn:",
    (df.groupby("clean_text")["label"].nunique() > 1).sum()
)

Bài rỗng: 0
Thẻ HTML: 0
URL: 0
Ký tự điều khiển: 0

Nội dung trùng sau làm sạch:
test 201
train 98
Nội dung chung train/test: 123
Nội dung có nhiều nhãn: 0


In [12]:
from sklearn.model_selection import train_test_split

train_data = df[df["split"] == "train"].copy()
test_df = df[df["split"] == "test"].copy()

groups = train_data[["clean_text", "label"]].drop_duplicates("clean_text")

train_groups, dev_groups = train_test_split(
    groups,
    test_size=0.15,
    random_state=42,
    stratify=groups["label"]
)

is_dev = train_data["clean_text"].isin(dev_groups["clean_text"])

train_df = train_data.loc[~is_dev].copy()
dev_df = train_data.loc[is_dev].copy()

assert set(train_df["clean_text"]).isdisjoint(dev_df["clean_text"])
assert len(train_df) + len(dev_df) == 25000
assert len(test_df) == 25000

for name, part in [
    ("train", train_df),
    ("dev", dev_df),
    ("test", test_df)
]:
    print(f"\n{name}: {len(part)} bài")
    print(part["label"].value_counts().sort_index())

print("\nNội dung chung train/dev: 0")


train: 21247 bài
label
0    10623
1    10624
Name: count, dtype: int64

dev: 3753 bài
label
0    1877
1    1876
Name: count, dtype: int64

test: 25000 bài
label
0    12500
1    12500
Name: count, dtype: int64

Nội dung chung train/dev: 0


In [13]:
from pathlib import Path
import json

output_dir = Path("/kaggle/working/task1")
output_dir.mkdir(exist_ok=True)

for name, part in [
    ("train", train_df),
    ("dev", dev_df),
    ("test", test_df)
]:
    part.to_csv(output_dir / f"{name}.csv", index=False)

split_ids = {
    "train": train_df["id"].tolist(),
    "dev": dev_df["id"].tolist(),
    "test": test_df["id"].tolist()
}

with open(output_dir / "split_ids.json", "w", encoding="utf-8") as f:
    json.dump(split_ids, f, ensure_ascii=False, indent=2)

print("Đã lưu:", output_dir)
print("Các file:", sorted(p.name for p in output_dir.iterdir()))

Đã lưu: /kaggle/working/task1
Các file: ['dev.csv', 'split_ids.json', 'test.csv', 'train.csv']
